In [1]:
# ============================================================================
# BLOCK 4: SENTIMENT ANALYSIS - ROMANIAN
# Input: 03_filtered_data_ro.pkl
# Output: 04_sentiment_data_ro.pkl
# Runtime: ~25-35 minutes on CPU
# ============================================================================
%run 00_setup_and_config.ipynb

c:\Users\andre\OneDrive\Desktop\2_Disertatie_RIPEMC_D.Flanja\Quantitative_Python\election_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Libraries loaded successfully
✓ PyTorch device: CPU
✓ Timestamp: 2026-05-29 04:10:50
✓ Directory structure verified
✓ Base directory: c:\Users\andre\OneDrive\Desktop\2_Disertatie_RIPEMC_D.Flanja\Quantitative_Python\YT_Analysis
✓ Model configuration loaded
  - Polish sentiment: eevvgg/bert-polish-sentiment-politics
  - Romanian sentiment: readerbench/ro-sentiment
  - Embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
  - BERTopic topics: 8
  - Batch size: 8
✓ Checkpoint utility functions loaded
✓ Text cleaning functions loaded
✓ Romanian stopwords: 385 words
✓ Polish stopwords: 511 words
✓ Visualization utility functions loaded
✓ Sentiment prediction function loaded
✓ SETUP AND CONFIGURATION COMPLETE

Ready for analysis. Next steps:
  1. Run Polish pipeline: 01_data_loading_pl.ipynb → 06_engagement_metrics_pl.ipynb
  2. Run Romanian pipeline: 01_data_loading_ro.ipynb → 06_engagement_metrics_ro.ipynb
  3. (Optional) Run comparative analysis: 08_comparative_ana

In [2]:
print('='*70)
print('BLOCK 4: SENTIMENT ANALYSIS - ROMANIAN')
print('='*70)

if check_checkpoint_exists('ro', '04_sentiment_data'):
    df_ro = load_checkpoint('ro', '04_sentiment_data')
    print('✓ Loading from sentiment checkpoint (skipping analysis)')
else:
    df_ro = load_checkpoint('ro', '03_filtered_data')
    if df_ro is None:
        raise FileNotFoundError('Run 02_text_cleaning_ro.ipynb first')

print(f'\nAnalyzing sentiment for {len(df_ro):,} Romanian comments...')
print(f'Model: {SENTIMENT_MODELS["ro"]}')
print(f'Batch size: {BATCH_SIZE}')

# Load model
print('\nLoading sentiment model...')

tokenizer = AutoTokenizer.from_pretrained(SENTIMENT_MODELS['ro'])
model = AutoModelForSequenceClassification.from_pretrained(SENTIMENT_MODELS['ro'])
model = model.to('cpu')
model.eval()

print('✓ Model loaded successfully')

# Predict
print('\nRunning sentiment analysis...')

texts = df_ro['clean_text'].fillna('').astype(str).tolist()
sentiment_ids = predict_sentiment_batch(texts, model, tokenizer, batch_size=BATCH_SIZE)

df_ro['sentiment_id'] = sentiment_ids
df_ro['sentiment'] = df_ro['sentiment_id'].apply(
    lambda x: SENTIMENT_LABELS['ro'][x] if x < len(SENTIMENT_LABELS['ro']) else 'Unknown'
)

# Distribution
print('\n' + '='*70)
print('SENTIMENT DISTRIBUTION - ROMANIAN')
print('='*70)

sentiment_counts = df_ro['sentiment'].value_counts()
sentiment_pct = 100 * sentiment_counts / len(df_ro)

for sentiment, count in sentiment_counts.items():
    pct = sentiment_pct[sentiment]
    print(f'  {sentiment:10} {count:>8,} ({pct:>5.1f}%)')

save_checkpoint(df_ro, 'ro', '04_sentiment_data')
update_pipeline_status('ro', 4, 'completed')

# Visualization
print('\nGenerating visualization...')
plot_sentiment_bar(df_ro, 'ro', 'sentiment')

# Export
sentiment_report = pd.DataFrame({
    'sentiment': sentiment_counts.index,
    'count': sentiment_counts.values,
    'percentage': sentiment_pct.values
})
sentiment_report.to_csv(OUTPUT_DIR / 'romanian' / 'ro_sentiment_report.csv', index=False)
print(f"\n✓ Saved: {OUTPUT_DIR / 'romanian' / 'ro_sentiment_report.csv'}")

print('\n' + '='*70)
print('✓ BLOCK 4 COMPLETE - SENTIMENT ANALYSIS DONE')
print('='*70)


BLOCK 4: SENTIMENT ANALYSIS - ROMANIAN
✓ Loading checkpoint: 03_filtered_data_ro.pkl

Analyzing sentiment for 6,600 Romanian comments...
Model: readerbench/ro-sentiment
Batch size: 8

Loading sentiment model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7851.94it/s]


✓ Model loaded successfully

Running sentiment analysis...


Predicting sentiment: 100%|██████████| 825/825 [24:22<00:00,  1.77s/it]



SENTIMENT DISTRIBUTION - ROMANIAN
  Negative      4,774 ( 72.3%)
  Positive      1,826 ( 27.7%)
✓ Checkpoint saved: 04_sentiment_data_ro.pkl
✓ Pipeline status updated: ro - Block 4 - completed

Generating visualization...
✓ Saved: outputs\ro\visualizations\ro_sentiment_bar.png

✓ Saved: outputs\romanian\ro_sentiment_report.csv

✓ BLOCK 4 COMPLETE - SENTIMENT ANALYSIS DONE
